# 02 — Exploratory Sentinel-1 Flood / Debris Change Mapping

This notebook creates an initial candidate flood/debris-related surface-change
mask for the 26 August 2026 Rasuwa–Bhote Koshi flash flood in Nepal.

The method compares a same-orbit Sentinel-1 baseline composite against the
5 September 2026 post-event Sentinel-1 scene.

**Method:** VV backscatter change + spatial smoothing + terrain masking +
permanent-water masking + connected-pixel filtering

**Primary sensor:** Sentinel-1 GRD  
**Geometry:** Descending pass, relative orbit 19  
**Status:** Exploratory — not a validated flood-extent product

In [1]:
from pathlib import Path
import sys

import ee
import geemap
import pandas as pd
from IPython.display import display

# ---------------------------------------------------------------------
# Project / output paths
# ---------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()

# If this notebook is run from a notebooks/ directory, move one level up.
if PROJECT_ROOT.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Earth Engine
# ---------------------------------------------------------------------
EE_PROJECT_ID = "nepal-flood-sentinel"

try:
    ee.Initialize(project=EE_PROJECT_ID)
except Exception as exc:
    raise RuntimeError(
        "Earth Engine initialization failed. Make sure you are authenticated "
        f"and have access to project '{EE_PROJECT_ID}'."
    ) from exc

print("Earth Engine initialized.")
print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine initialized.
Project root: /Users/nishantanepal/Desktop/Data Analytics/nepal flood sentinel damage
Python executable: /Users/nishantanepal/Desktop/Data Analytics/nepal flood sentinel damage/.venv/bin/python


In [2]:
# ---------------------------------------------------------------------
# Analysis configuration
# ---------------------------------------------------------------------

EVENT_NAME = "Rasuwa–Bhote Koshi flash flood"
EVENT_DATE = "2026-08-26"

# Pre-event search window.
# The final baseline scenes are selected automatically from this interval.
BASELINE_START = "2026-05-01"
BASELINE_END = EVENT_DATE  # filterDate end is exclusive

# Post-event search window.
POST_EVENT_START = EVENT_DATE
POST_EVENT_END = "2026-09-20"

# Corrected AOI:
# Covers Rasuwagadhi and the documented upper Lhende source area.
# This is intentionally a regional analysis box, not a flood polygon.
AOI_BBOX = {
    "min_lon": 85.25,
    "min_lat": 28.05,
    "max_lon": 85.65,
    "max_lat": 28.35,
}

# Reference locations used only for geographic validation on the map.
REFERENCE_POINTS = {
    "Rasuwagadhi": {
        "lon": 85.38022,
        "lat": 28.27722,
    },
    "Approx. upper Lhende source area": {
        "lon": 85.528159,
        "lat": 28.288708,
    },
}

# Sentinel-1 selection settings.
MIN_AOI_COVERAGE = 0.90
MIN_BASELINE_SCENES = 3
MAX_BASELINE_SCENES = 5
PREFER_SAME_PLATFORM = True

# Processing parameters.
SMOOTH_RADIUS_M = 20
VV_CHANGE_THRESHOLD_DB = -3.0
BROAD_CHANGE_THRESHOLD_DB = 3.0
MAX_FLOOD_SLOPE_DEGREES = 15
MAX_BROAD_CHANGE_SLOPE_DEGREES = 45
PERMANENT_WATER_OCCURRENCE = 90
MIN_CONNECTED_PIXELS = 8
MAX_PIXELS = 1e10

print(f"Event: {EVENT_NAME}")
print(f"Event date: {EVENT_DATE}")
print(f"AOI: {AOI_BBOX}")

Event: Rasuwa–Bhote Koshi flash flood
Event date: 2026-08-26
AOI: {'min_lon': 85.25, 'min_lat': 28.05, 'max_lon': 85.65, 'max_lat': 28.35}


In [3]:
# ---------------------------------------------------------------------
# AOI geometry
# ---------------------------------------------------------------------

analysis_aoi = ee.Geometry.Rectangle(
    [
        AOI_BBOX["min_lon"],
        AOI_BBOX["min_lat"],
        AOI_BBOX["max_lon"],
        AOI_BBOX["max_lat"],
    ],
    proj="EPSG:4326",
    geodesic=False,
)

aoi_centroid = analysis_aoi.centroid(maxError=10).coordinates().getInfo()
AOI_CENTER = [aoi_centroid[1], aoi_centroid[0]]

aoi_area_km2 = ee.Number(analysis_aoi.area(maxError=10)).divide(1e6).getInfo()

print("Analysis AOI:")
print(analysis_aoi.getInfo())
print(f"AOI center: {AOI_CENTER}")
print(f"AOI area: {aoi_area_km2:.2f} km²")

Analysis AOI:
{'geodesic': False, 'type': 'Polygon', 'coordinates': [[[85.25, 28.05], [85.65, 28.05], [85.65, 28.35], [85.25, 28.35], [85.25, 28.05]]]}
AOI center: [28.200002497763045, 85.45000000000016]
AOI area: 1307.61 km²


In [4]:
# ---------------------------------------------------------------------
# Geographic sanity-check map
# ---------------------------------------------------------------------

aoi_map = geemap.Map(center=AOI_CENTER, zoom=10)

aoi_map.addLayer(
    analysis_aoi,
    {
        "color": "yellow",
        "fillColor": "00000000",
    },
    "Corrected analysis AOI",
)

for name, point in REFERENCE_POINTS.items():
    feature = ee.Feature(
        ee.Geometry.Point([point["lon"], point["lat"]]),
        {"name": name},
    )
    aoi_map.addLayer(
        feature,
        {"color": "red"},
        name,
    )

aoi_map


Map(center=[28.200002497763045, 85.45000000000016], controls=(WidgetControl(options=['position', 'transparent_…

In [5]:
# ---------------------------------------------------------------------
# Sentinel-1 query helpers
# ---------------------------------------------------------------------

S1_COLLECTION_ID = "COPERNICUS/S1_GRD"

def base_sentinel1_collection(start_date, end_date):
    """Return homogeneous Sentinel-1 IW VV/VH scenes intersecting the AOI."""
    return (
        ee.ImageCollection(S1_COLLECTION_ID)
        .filterBounds(analysis_aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VV",
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VH",
            )
        )
    )


def add_aoi_coverage(image):
    """Attach the fraction of the AOI covered by each scene footprint."""
    aoi_area = analysis_aoi.area(maxError=10)

    overlap_area = (
        image.geometry()
        .intersection(analysis_aoi, maxError=10)
        .area(maxError=10)
    )

    coverage = ee.Number(overlap_area).divide(aoi_area)

    return image.set(
        "aoi_coverage_fraction",
        coverage,
    )


In [9]:
# ---------------------------------------------------------------------
# Metadata helper — FIXED
# ---------------------------------------------------------------------

def sentinel1_dataframe(collection):
    """Convert selected Sentinel-1 metadata to a pandas DataFrame."""

    collection = collection.map(add_aoi_coverage)

    def image_to_feature(image):
        return ee.Feature(
            None,
            {
                "scene_index": image.get("system:index"),
                "system_time_start": image.get("system:time_start"),
                "orbit_pass": image.get("orbitProperties_pass"),
                "relative_orbit": image.get("relativeOrbitNumber_start"),
                "platform": image.get("platform_number"),
                "resolution_meters": image.get("resolution_meters"),
                "aoi_coverage_fraction": image.get(
                    "aoi_coverage_fraction"
                ),
            },
        )

    features = collection.map(image_to_feature)

    info = features.getInfo()
    records = info.get("features", [])

    columns = [
        "scene_index",
        "system_time_start",
        "orbit_pass",
        "relative_orbit",
        "platform",
        "resolution_meters",
        "aoi_coverage_fraction",
        "acquisition_time",
    ]

    if not records:
        return pd.DataFrame(columns=columns)

    dataframe = pd.DataFrame(
        [item["properties"] for item in records]
    )

    dataframe["acquisition_time"] = pd.to_datetime(
        dataframe["system_time_start"],
        unit="ms",
        utc=True,
    )

    dataframe["aoi_coverage_fraction"] = pd.to_numeric(
        dataframe["aoi_coverage_fraction"],
        errors="coerce",
    )

    dataframe["relative_orbit"] = pd.to_numeric(
        dataframe["relative_orbit"],
        errors="coerce",
    )

    return (
        dataframe
        .sort_values("acquisition_time")
        .reset_index(drop=True)
    )

In [10]:
# ---------------------------------------------------------------------
# Load all pre- and post-event candidates
# ---------------------------------------------------------------------

s1_baseline_all = base_sentinel1_collection(
    BASELINE_START,
    BASELINE_END,
)

s1_post_all = base_sentinel1_collection(
    POST_EVENT_START,
    POST_EVENT_END,
)

baseline_inventory = sentinel1_dataframe(s1_baseline_all)
post_inventory = sentinel1_dataframe(s1_post_all)

print("All pre-event Sentinel-1 candidates:", len(baseline_inventory))
print("All post-event Sentinel-1 candidates:", len(post_inventory))

if post_inventory.empty:
    raise RuntimeError(
        "No post-event Sentinel-1 IW VV/VH scenes intersect the corrected AOI "
        f"between {POST_EVENT_START} and {POST_EVENT_END}."
    )


All pre-event Sentinel-1 candidates: 30
All post-event Sentinel-1 candidates: 6


In [11]:
# ---------------------------------------------------------------------
# Inspect complete scene inventory — FIXED
# ---------------------------------------------------------------------

inventory_columns = [
    "acquisition_time",
    "scene_index",
    "platform",
    "orbit_pass",
    "relative_orbit",
    "resolution_meters",
    "aoi_coverage_fraction",
]

print("Pre-event candidates:")

if baseline_inventory.empty:
    print("No pre-event Sentinel-1 candidates found.")
else:
    display(
        baseline_inventory[inventory_columns]
        .style.format(
            {
                "aoi_coverage_fraction": "{:.3f}",
            }
        )
    )

print("Post-event candidates:")

if post_inventory.empty:
    print("No post-event Sentinel-1 candidates found.")
else:
    display(
        post_inventory[inventory_columns]
        .style.format(
            {
                "aoi_coverage_fraction": "{:.3f}",
            }
        )
    )

Pre-event candidates:


,acquisition_time,scene_index,platform,orbit_pass,relative_orbit,resolution_meters,aoi_coverage_fraction
0,2026-05-01 00:19:24+00:00,S1A_IW_GRDH_1SDV_20260501T001924_20260501T001949_064316_08190C_F57F,A,DESCENDING,19,10,1.000
1,2026-05-05 12:22:21+00:00,S1A_IW_GRDH_1SDV_20260505T122221_20260505T122246_064382_081B72_E59F,A,ASCENDING,85,10,1.000
2,2026-05-08 00:11:16+00:00,S1A_IW_GRDH_1SDV_20260508T001116_20260508T001141_064418_081CD6_8331,A,DESCENDING,121,10,0.672
3,2026-05-13 00:19:23+00:00,S1A_IW_GRDH_1SDV_20260513T001923_20260513T001948_064491_081F60_4E7B,A,DESCENDING,19,10,1.000
4,2026-05-17 12:22:20+00:00,S1A_IW_GRDH_1SDV_20260517T122220_20260517T122245_064557_08219F_2331,A,ASCENDING,85,10,1.000
5,2026-05-20 00:11:16+00:00,S1A_IW_GRDH_1SDV_20260520T001116_20260520T001141_064593_0822ED_B174,A,DESCENDING,121,10,0.675
6,2026-05-29 12:22:20+00:00,S1A_IW_GRDH_1SDV_20260529T122220_20260529T122245_064732_0827C0_A4CC,A,ASCENDING,85,10,1.000
7,2026-06-01 00:11:15+00:00,S1A_IW_GRDH_1SDV_20260601T001115_20260601T001140_064768_08290F_8158,A,DESCENDING,121,10,0.676
8,2026-06-06 00:19:22+00:00,S1A_IW_GRDH_1SDV_20260606T001922_20260606T001947_064841_082BA1_CA2B,A,DESCENDING,19,10,1.000
9,2026-06-10 12:22:19+00:00,S1A_IW_GRDH_1SDV_20260610T122219_20260610T122244_064907_082DE1_2D66,A,ASCENDING,85,10,1.000


Post-event candidates:


,acquisition_time,scene_index,platform,orbit_pass,relative_orbit,resolution_meters,aoi_coverage_fraction
0,2026-08-28 12:21:41+00:00,S1D_IW_GRDH_1SDV_20260828T122141_20260828T122206_004326_007FA4_01B4,D,ASCENDING,85,10,1.000
1,2026-08-31 00:10:37+00:00,S1D_IW_GRDH_1SDV_20260831T001037_20260831T001102_004362_0080EC_2C5B,D,DESCENDING,121,10,0.670
2,2026-09-05 00:18:44+00:00,S1D_IW_GRDH_1SDV_20260905T001844_20260905T001909_004435_00837B_2F0C,D,DESCENDING,19,10,1.000
3,2026-09-09 12:21:41+00:00,S1D_IW_GRDH_1SDV_20260909T122141_20260909T122206_004501_0085C5_10A0,D,ASCENDING,85,10,1.000
4,2026-09-12 00:10:38+00:00,S1D_IW_GRDH_1SDV_20260912T001038_20260912T001103_004537_008704_4E96,D,DESCENDING,121,10,0.671
5,2026-09-17 00:18:45+00:00,S1D_IW_GRDH_1SDV_20260917T001845_20260917T001910_004610_008985_92D6,D,DESCENDING,19,10,1.000


In [12]:
# ---------------------------------------------------------------------
# Select one explicit post-event scene — FIXED
# ---------------------------------------------------------------------

post_full = post_inventory[
    post_inventory["aoi_coverage_fraction"] >= MIN_AOI_COVERAGE
].copy()

if post_full.empty:
    print(
        "Warning: no post-event scene covers at least "
        f"{MIN_AOI_COVERAGE:.0%} of the AOI."
    )

    print(
        "Falling back to the scene with the greatest AOI coverage."
    )

    post_pool = post_inventory.sort_values(
        [
            "aoi_coverage_fraction",
            "acquisition_time",
        ],
        ascending=[
            False,
            True,
        ],
    )

else:
    post_pool = post_full.sort_values(
        [
            "acquisition_time",
            "aoi_coverage_fraction",
        ],
        ascending=[
            True,
            False,
        ],
    )

selected_post_row = post_pool.iloc[0]

POST_SCENE_INDEX = selected_post_row["scene_index"]
POST_PASS = selected_post_row["orbit_pass"]
POST_RELATIVE_ORBIT = int(
    selected_post_row["relative_orbit"]
)
POST_PLATFORM = selected_post_row["platform"]
POST_TIMESTAMP = selected_post_row["acquisition_time"]

post_image = ee.Image(
    s1_post_all
    .filter(
        ee.Filter.eq(
            "system:index",
            POST_SCENE_INDEX,
        )
    )
    .first()
)

print("Selected post-event scene")
print("-------------------------")
print(f"Acquisition: {POST_TIMESTAMP}")
print(f"Index: {POST_SCENE_INDEX}")
print(f"Platform: {POST_PLATFORM}")
print(f"Pass: {POST_PASS}")
print(f"Relative orbit: {POST_RELATIVE_ORBIT}")
print(
    "AOI coverage:",
    f"{selected_post_row['aoi_coverage_fraction']:.1%}",
)

Selected post-event scene
-------------------------
Acquisition: 2026-08-28 12:21:41+00:00
Index: S1D_IW_GRDH_1SDV_20260828T122141_20260828T122206_004326_007FA4_01B4
Platform: D
Pass: ASCENDING
Relative orbit: 85
AOI coverage: 100.0%


In [13]:
# ---------------------------------------------------------------------
# Match baseline geometry to selected post-event scene — FIXED
# ---------------------------------------------------------------------

baseline_geometry_matched = (
    s1_baseline_all
    .filter(
        ee.Filter.eq(
            "orbitProperties_pass",
            POST_PASS,
        )
    )
    .filter(
        ee.Filter.eq(
            "relativeOrbitNumber_start",
            POST_RELATIVE_ORBIT,
        )
    )
)

baseline_geometry_inventory = sentinel1_dataframe(
    baseline_geometry_matched
)

baseline_good_coverage = baseline_geometry_inventory[
    baseline_geometry_inventory["aoi_coverage_fraction"]
    >= MIN_AOI_COVERAGE
].copy()

if baseline_good_coverage.empty:
    raise RuntimeError(
        "No pre-event scenes with matching pass/orbit cover enough "
        "of the corrected AOI. Inspect the inventory and consider "
        "changing the AOI or MIN_AOI_COVERAGE."
    )


# Prefer same Sentinel-1 satellite/platform when possible.
same_platform = baseline_good_coverage[
    baseline_good_coverage["platform"]
    == POST_PLATFORM
].copy()


if (
    PREFER_SAME_PLATFORM
    and len(same_platform) >= MIN_BASELINE_SCENES
):
    baseline_pool = same_platform

    BASELINE_PLATFORM_MODE = (
        f"same platform ({POST_PLATFORM})"
    )

else:
    baseline_pool = baseline_good_coverage

    BASELINE_PLATFORM_MODE = (
        "same pass/orbit, mixed platform if needed"
    )


if len(baseline_pool) < MIN_BASELINE_SCENES:
    raise RuntimeError(
        f"Only {len(baseline_pool)} adequate matching "
        f"baseline scenes were found; at least "
        f"{MIN_BASELINE_SCENES} are required."
    )


# Select the most recent matching scenes before the event.
selected_baseline_rows = (
    baseline_pool
    .sort_values("acquisition_time")
    .tail(MAX_BASELINE_SCENES)
    .copy()
)


selected_baseline_indices = (
    selected_baseline_rows["scene_index"]
    .tolist()
)


s1_baseline_selected = (
    baseline_geometry_matched
    .filter(
        ee.Filter.inList(
            "system:index",
            selected_baseline_indices,
        )
    )
)


print(
    "Baseline selection mode:",
    BASELINE_PLATFORM_MODE,
)

print(
    "Selected baseline scene count:",
    len(selected_baseline_rows),
)

display(
    selected_baseline_rows[
        inventory_columns
    ].style.format(
        {
            "aoi_coverage_fraction": "{:.3f}",
        }
    )
)

Baseline selection mode: same platform (D)
Selected baseline scene count: 5


,acquisition_time,scene_index,platform,orbit_pass,relative_orbit,resolution_meters,aoi_coverage_fraction
5,2026-06-29 12:21:38+00:00,S1D_IW_GRDH_1SDV_20260629T122138_20260629T122203_003451_006174_F84B,D,ASCENDING,85,10,1.000
6,2026-07-11 12:21:39+00:00,S1D_IW_GRDH_1SDV_20260711T122139_20260711T122204_003626_00675A_805D,D,ASCENDING,85,10,1.000
7,2026-07-23 12:21:40+00:00,S1D_IW_GRDH_1SDV_20260723T122140_20260723T122205_003801_006D5D_CDAA,D,ASCENDING,85,10,1.000
8,2026-08-04 12:21:40+00:00,S1D_IW_GRDH_1SDV_20260804T122140_20260804T122205_003976_00737A_6F58,D,ASCENDING,85,10,1.000
9,2026-08-16 12:21:41+00:00,S1D_IW_GRDH_1SDV_20260816T122141_20260816T122206_004151_007980_C3AB,D,ASCENDING,85,10,1.000


In [15]:
# ---------------------------------------------------------------------
# Scene-footprint validation map — FIXED
# ---------------------------------------------------------------------

footprint_map = geemap.Map(
    center=AOI_CENTER,
    zoom=9,
)


# ---------------------------------------------------------------------
# AOI
# ---------------------------------------------------------------------

footprint_map.addLayer(
    analysis_aoi,
    {
        "color": "yellow",
        "fillColor": "00000000",
    },
    "Analysis AOI",
)


# ---------------------------------------------------------------------
# Convert selected baseline ImageCollection -> FeatureCollection
#
# IMPORTANT:
# ImageCollection.map() must return Images.
# Therefore we convert the collection to a List first, then map the
# images into Features.
# ---------------------------------------------------------------------

baseline_count = s1_baseline_selected.size()

baseline_image_list = s1_baseline_selected.toList(
    baseline_count
)


baseline_footprints = ee.FeatureCollection(
    ee.List.sequence(
        0,
        baseline_count.subtract(1),
    ).map(
        lambda i: ee.Feature(
            ee.Image(
                baseline_image_list.get(i)
            ).geometry(),
            {
                "scene": ee.Image(
                    baseline_image_list.get(i)
                ).get("system:index"),
            },
        )
    )
)


# ---------------------------------------------------------------------
# Post-event footprint
# ---------------------------------------------------------------------

post_footprint = ee.FeatureCollection(
    [
        ee.Feature(
            post_image.geometry(),
            {
                "scene": POST_SCENE_INDEX,
            },
        )
    ]
)


# ---------------------------------------------------------------------
# Paint footprints as raster images.
#
# This is more reliable with geemap than directly passing the
# FeatureCollections.
# ---------------------------------------------------------------------

baseline_footprint_image = (
    ee.Image()
    .byte()
    .paint(
        featureCollection=baseline_footprints,
        color=1,
        width=2,
    )
)


post_footprint_image = (
    ee.Image()
    .byte()
    .paint(
        featureCollection=post_footprint,
        color=1,
        width=3,
    )
)


# ---------------------------------------------------------------------
# Add footprints
# ---------------------------------------------------------------------

footprint_map.addLayer(
    baseline_footprint_image,
    {
        "palette": ["00ff00"],
    },
    "Selected baseline footprints",
)


footprint_map.addLayer(
    post_footprint_image,
    {
        "palette": ["ff0000"],
    },
    "Selected post-event footprint",
)


# ---------------------------------------------------------------------
# Reference locations
# ---------------------------------------------------------------------

for name, point in REFERENCE_POINTS.items():

    reference_point = ee.Geometry.Point(
        [
            point["lon"],
            point["lat"],
        ]
    )

    # Buffer the point slightly so it is visible as a small circle.
    reference_marker = (
        ee.Image()
        .byte()
        .paint(
            ee.FeatureCollection(
                [
                    ee.Feature(
                        reference_point.buffer(300)
                    )
                ]
            ),
            color=1,
        )
    )

    footprint_map.addLayer(
        reference_marker,
        {
            "palette": ["ffffff"],
        },
        name,
    )


# ---------------------------------------------------------------------
# Center map on AOI
# ---------------------------------------------------------------------

footprint_map.centerObject(
    analysis_aoi,
    9,
)


footprint_map

Map(center=[28.199978270975766, 85.45000000000016], controls=(WidgetControl(options=['position', 'transparent_…

In [16]:
print("POST-EVENT SCENE")
print("----------------")
print("Date:", POST_TIMESTAMP)
print("Platform:", POST_PLATFORM)
print("Pass:", POST_PASS)
print("Relative orbit:", POST_RELATIVE_ORBIT)
print("Scene:", POST_SCENE_INDEX)

print()

print("BASELINE SCENES")
print("---------------")

display(
    selected_baseline_rows[
        [
            "acquisition_time",
            "platform",
            "orbit_pass",
            "relative_orbit",
            "aoi_coverage_fraction",
        ]
    ]
)

POST-EVENT SCENE
----------------
Date: 2026-08-28 12:21:41+00:00
Platform: D
Pass: ASCENDING
Relative orbit: 85
Scene: S1D_IW_GRDH_1SDV_20260828T122141_20260828T122206_004326_007FA4_01B4

BASELINE SCENES
---------------


,acquisition_time,platform,orbit_pass,relative_orbit,aoi_coverage_fraction
5,2026-06-29 12:21:38+00:00,D,ASCENDING,85,1
6,2026-07-11 12:21:39+00:00,D,ASCENDING,85,1
7,2026-07-23 12:21:40+00:00,D,ASCENDING,85,1
8,2026-08-04 12:21:40+00:00,D,ASCENDING,85,1
9,2026-08-16 12:21:41+00:00,D,ASCENDING,85,1


In [17]:
# ---------------------------------------------------------------------
# Baseline composite, post-event image, and SAR change
# ---------------------------------------------------------------------

baseline_s1 = (
    s1_baseline_selected
    .select(["VV", "VH"])
    .median()
    .clip(analysis_aoi)
)

post_s1 = (
    post_image
    .select(["VV", "VH"])
    .clip(analysis_aoi)
)

vv_pre = baseline_s1.select("VV").rename("VV_pre")
vv_post = post_s1.select("VV").rename("VV_post")

vh_pre = baseline_s1.select("VH").rename("VH_pre")
vh_post = post_s1.select("VH").rename("VH_post")

vv_change = (
    vv_post
    .subtract(vv_pre)
    .rename("VV_change_db")
)

vh_change = (
    vh_post
    .subtract(vh_pre)
    .rename("VH_change_db")
)

print("Baseline bands:", baseline_s1.bandNames().getInfo())
print("Post-event bands:", post_s1.bandNames().getInfo())
print("VV change band:", vv_change.bandNames().getInfo())
print("VH change band:", vh_change.bandNames().getInfo())


Baseline bands: ['VV', 'VH']
Post-event bands: ['VV', 'VH']
VV change band: ['VV_change_db']
VH change band: ['VH_change_db']


In [18]:
# ---------------------------------------------------------------------
# Conservative spatial smoothing
# ---------------------------------------------------------------------

# Smooth pre and post images separately, then calculate the dB difference.
# A 20 m radius is less aggressive than the original 30 m radius and helps
# preserve narrow mountain-river features.

vv_pre_smooth = (
    vv_pre
    .focal_median(
        radius=SMOOTH_RADIUS_M,
        units="meters",
    )
    .rename("VV_pre_smooth")
)

vv_post_smooth = (
    vv_post
    .focal_median(
        radius=SMOOTH_RADIUS_M,
        units="meters",
    )
    .rename("VV_post_smooth")
)

vv_change_smooth = (
    vv_post_smooth
    .subtract(vv_pre_smooth)
    .rename("VV_change_smooth_db")
)

print(
    "Smoothed VV change created with radius:",
    f"{SMOOTH_RADIUS_M} m",
)


Smoothed VV change created with radius: 20 m


In [19]:
# ---------------------------------------------------------------------
# Terrain masks
# ---------------------------------------------------------------------

dem = (
    ee.Image("USGS/SRTMGL1_003")
    .select("elevation")
    .clip(analysis_aoi)
)

slope = (
    ee.Terrain.slope(dem)
    .rename("slope_degrees")
)

flood_slope_mask = (
    slope
    .lte(MAX_FLOOD_SLOPE_DEGREES)
    .rename("flood_slope_mask")
)

broad_change_slope_mask = (
    slope
    .lte(MAX_BROAD_CHANGE_SLOPE_DEGREES)
    .rename("broad_change_slope_mask")
)

print(
    "Flood-like maximum slope:",
    f"{MAX_FLOOD_SLOPE_DEGREES}°",
)

print(
    "Broader surface-change maximum slope:",
    f"{MAX_BROAD_CHANGE_SLOPE_DEGREES}°",
)


Flood-like maximum slope: 15°
Broader surface-change maximum slope: 45°


In [20]:
# ---------------------------------------------------------------------
# Historical permanent-water mask — corrected
# ---------------------------------------------------------------------

jrc_occurrence = (
    ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
    .select("occurrence")
    .clip(analysis_aoi)
)

# IMPORTANT FIX:
# The JRC occurrence band is masked where water was never observed.
# unmask(0) explicitly treats those pixels as 0% historical water occurrence.
jrc_occurrence_unmasked = jrc_occurrence.unmask(0)

permanent_water_mask = (
    jrc_occurrence_unmasked
    .gte(PERMANENT_WATER_OCCURRENCE)
    .rename("historical_permanent_water")
)

non_permanent_water_mask = (
    permanent_water_mask
    .Not()
    .rename("non_permanent_water")
)

print(
    "Historical permanent water defined as JRC occurrence >=",
    f"{PERMANENT_WATER_OCCURRENCE}%.",
)

print(
    "Pixels with no JRC water observation are explicitly treated as non-water "
    "using unmask(0)."
)


Historical permanent water defined as JRC occurrence >= 90%.
Pixels with no JRC water observation are explicitly treated as non-water using unmask(0).


In [21]:
# ---------------------------------------------------------------------
# Raw flood-like candidate
# ---------------------------------------------------------------------

# This mask targets strong negative VV change on relatively gentle terrain,
# outside historical permanent water.
#
# It should be interpreted as "flood-like SAR change", not definitive flooding.

raw_flood_candidate = (
    vv_change_smooth
    .lte(VV_CHANGE_THRESHOLD_DB)
    .And(flood_slope_mask)
    .And(non_permanent_water_mask)
    .rename("raw_flood_candidate")
)

print(
    "Flood-like VV threshold:",
    f"<= {VV_CHANGE_THRESHOLD_DB} dB",
)


Flood-like VV threshold: <= -3.0 dB


In [22]:
# ---------------------------------------------------------------------
# Broader SAR surface-change candidate
# ---------------------------------------------------------------------

# Debris/channel disturbance can cause either increases or decreases in SAR
# backscatter. Therefore, do not use only a negative-change rule for this layer.
#
# This remains a generic "surface-change" candidate because SAR alone does not
# prove that a changed pixel is debris.

raw_surface_change_candidate = (
    vv_change_smooth
    .abs()
    .gte(BROAD_CHANGE_THRESHOLD_DB)
    .And(broad_change_slope_mask)
    .rename("raw_surface_change_candidate")
)

print(
    "Broader absolute VV-change threshold:",
    f">= {BROAD_CHANGE_THRESHOLD_DB} dB",
)


Broader absolute VV-change threshold: >= 3.0 dB


In [23]:
# ---------------------------------------------------------------------
# Connected-pixel filtering — corrected
# ---------------------------------------------------------------------

def remove_small_components(binary_image, min_pixels):
    """Keep only connected candidate components with at least min_pixels."""

    # selfMask() removes zero-valued background before connected-component counting.
    candidate_only = binary_image.selfMask()

    connected = candidate_only.connectedPixelCount(
        maxSize=1024,
        eightConnected=True,
    )

    return (
        candidate_only
        .updateMask(connected.gte(min_pixels))
        .selfMask()
    )


flood_candidate_mask = (
    remove_small_components(
        raw_flood_candidate,
        MIN_CONNECTED_PIXELS,
    )
    .rename("flood_candidate_mask")
)

surface_change_mask = (
    remove_small_components(
        raw_surface_change_candidate,
        MIN_CONNECTED_PIXELS,
    )
    .rename("surface_change_mask")
)

print(
    "Minimum connected candidate pixels:",
    MIN_CONNECTED_PIXELS,
)


Minimum connected candidate pixels: 8


In [24]:
# ---------------------------------------------------------------------
# Visualization
# ---------------------------------------------------------------------

sar_vis = {
    "min": -25,
    "max": 0,
}

# Wider than the classification threshold so strong changes do not all saturate.
change_vis = {
    "min": -8,
    "max": 8,
    "palette": [
        "67001f",
        "d6604d",
        "f7f7f7",
        "4393c3",
        "053061",
    ],
}

flood_mask_vis = {
    "palette": ["00ffff"],
}

surface_change_vis = {
    "palette": ["ff00ff"],
}

water_vis = {
    "palette": ["0000ff"],
}

slope_vis = {
    "min": 0,
    "max": 45,
    "palette": [
        "1a9850",
        "fee08b",
        "d73027",
    ],
}

result_map = geemap.Map(
    center=AOI_CENTER,
    zoom=10,
)

result_map.addLayer(
    vv_pre_smooth,
    sar_vis,
    "Baseline VV (smoothed)",
    False,
)

result_map.addLayer(
    vv_post_smooth,
    sar_vis,
    f"Post-event VV — {POST_TIMESTAMP.date()} (smoothed)",
    False,
)

result_map.addLayer(
    vv_change_smooth,
    change_vis,
    "VV change (post − baseline), dB",
    True,
)

result_map.addLayer(
    flood_candidate_mask,
    flood_mask_vis,
    "Flood-like negative-change candidate",
    True,
)

result_map.addLayer(
    surface_change_mask,
    surface_change_vis,
    "Broader SAR surface-change candidate",
    False,
)

result_map.addLayer(
    permanent_water_mask.selfMask(),
    water_vis,
    "Historical permanent water (JRC >= 90%)",
    False,
)

result_map.addLayer(
    slope,
    slope_vis,
    "Slope (degrees)",
    False,
)

result_map.addLayer(
    analysis_aoi,
    {
        "color": "yellow",
        "fillColor": "00000000",
    },
    "Analysis AOI",
    True,
)

for name, point in REFERENCE_POINTS.items():
    result_map.addLayer(
        ee.Feature(
            ee.Geometry.Point([point["lon"], point["lat"]]),
            {"name": name},
        ),
        {"color": "white"},
        name,
        True,
    )

result_map


Map(center=[28.200002497763045, 85.45000000000016], controls=(WidgetControl(options=['position', 'transparent_…

In [25]:
# ---------------------------------------------------------------------
# Candidate-area calculation
# ---------------------------------------------------------------------

def masked_area_km2(mask_image, geometry, scale=10):
    area_image = (
        ee.Image.pixelArea()
        .updateMask(mask_image)
        .rename("area_m2")
    )

    result = area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geometry,
        scale=scale,
        maxPixels=MAX_PIXELS,
        bestEffort=False,
    )

    area_m2 = ee.Number(
        ee.Algorithms.If(
            result.contains("area_m2"),
            result.get("area_m2"),
            0,
        )
    )

    return area_m2.divide(1e6)


flood_candidate_area_km2 = masked_area_km2(
    flood_candidate_mask,
    analysis_aoi,
)

surface_change_area_km2 = masked_area_km2(
    surface_change_mask,
    analysis_aoi,
)

print(
    "Flood-like candidate area (km²):",
    flood_candidate_area_km2.getInfo(),
)

print(
    "Broader SAR surface-change candidate area (km²):",
    surface_change_area_km2.getInfo(),
)


Flood-like candidate area (km²): 1.3768585937042237
Broader SAR surface-change candidate area (km²): 34.595594449699405


In [26]:
# ---------------------------------------------------------------------
# Simple threshold sensitivity check
# ---------------------------------------------------------------------

# This does not validate the result; it shows whether the mapped flood-like area
# is extremely sensitive to small threshold changes.

thresholds_db = [-2.0, -3.0, -4.0]

sensitivity_rows = []

for threshold in thresholds_db:
    raw = (
        vv_change_smooth
        .lte(threshold)
        .And(flood_slope_mask)
        .And(non_permanent_water_mask)
    )

    filtered = remove_small_components(
        raw,
        MIN_CONNECTED_PIXELS,
    )

    area = masked_area_km2(
        filtered,
        analysis_aoi,
    ).getInfo()

    sensitivity_rows.append(
        {
            "VV_threshold_dB": threshold,
            "candidate_area_km2": area,
        }
    )

sensitivity_df = pd.DataFrame(sensitivity_rows)
display(sensitivity_df)


,VV_threshold_dB,candidate_area_km2
0,-2.0,6.165451
1,-3.0,1.376859
2,-4.0,0.362962


In [27]:
# ---------------------------------------------------------------------
# Save interactive map
# ---------------------------------------------------------------------

html_path = (
    FIGURES_DIR
    / "phase2_corrected_flood_surface_change.html"
)

result_map.to_html(str(html_path))

print(
    "Saved corrected interactive map to:",
    html_path,
)


Saved corrected interactive map to: /Users/nishantanepal/Desktop/Data Analytics/nepal flood sentinel damage/figures/phase2_corrected_flood_surface_change.html


In [28]:
# ---------------------------------------------------------------------
# Final reproducibility summary
# ---------------------------------------------------------------------

print("Corrected exploratory mapping summary")
print("-------------------------------------")
print(f"Event: {EVENT_NAME}")
print(f"Event date: {EVENT_DATE}")
print(f"AOI: {AOI_BBOX}")
print(f"AOI area: {aoi_area_km2:.2f} km²")
print()

print("Selected post-event scene")
print(f"  Acquisition: {POST_TIMESTAMP}")
print(f"  Platform: {POST_PLATFORM}")
print(f"  Pass: {POST_PASS}")
print(f"  Relative orbit: {POST_RELATIVE_ORBIT}")
print(f"  Index: {POST_SCENE_INDEX}")
print()

print("Baseline")
print(f"  Selection mode: {BASELINE_PLATFORM_MODE}")
print(f"  Scene count: {len(selected_baseline_rows)}")
print(
    "  Dates:",
    [
        ts.strftime("%Y-%m-%d")
        for ts in selected_baseline_rows["acquisition_time"]
    ],
)
print()

print("Processing")
print(f"  Smoothing radius: {SMOOTH_RADIUS_M} m")
print(f"  Flood-like VV threshold: <= {VV_CHANGE_THRESHOLD_DB} dB")
print(f"  Flood-like maximum slope: {MAX_FLOOD_SLOPE_DEGREES}°")
print(
    "  Historical permanent-water exclusion:",
    f"JRC occurrence >= {PERMANENT_WATER_OCCURRENCE}%",
)
print(f"  Minimum connected pixels: {MIN_CONNECTED_PIXELS}")
print()

print(
    "Flood-like candidate area (km²):",
    flood_candidate_area_km2.getInfo(),
)

print(
    "Broader SAR surface-change candidate area (km²):",
    surface_change_area_km2.getInfo(),
)

print()
print(
    "Interpretation warning: these are exploratory SAR-change candidates. "
    "They require validation against optical imagery, mapped river/channel data, "
    "field observations, or an authoritative flood/debris product before being "
    "reported as event extent."
)


Corrected exploratory mapping summary
-------------------------------------
Event: Rasuwa–Bhote Koshi flash flood
Event date: 2026-08-26
AOI: {'min_lon': 85.25, 'min_lat': 28.05, 'max_lon': 85.65, 'max_lat': 28.35}
AOI area: 1307.61 km²

Selected post-event scene
  Acquisition: 2026-08-28 12:21:41+00:00
  Platform: D
  Pass: ASCENDING
  Relative orbit: 85
  Index: S1D_IW_GRDH_1SDV_20260828T122141_20260828T122206_004326_007FA4_01B4

Baseline
  Selection mode: same platform (D)
  Scene count: 5
  Dates: ['2026-06-29', '2026-07-11', '2026-07-23', '2026-08-04', '2026-08-16']

Processing
  Smoothing radius: 20 m
  Flood-like VV threshold: <= -3.0 dB
  Flood-like maximum slope: 15°
  Historical permanent-water exclusion: JRC occurrence >= 90%
  Minimum connected pixels: 8

Flood-like candidate area (km²): 1.3768585937042237
Broader SAR surface-change candidate area (km²): 34.595594449699405

Interpretation warning: these are exploratory SAR-change candidates. They require validation against 

In [29]:
diagnostic_map = geemap.Map(
    center=AOI_CENTER,
    zoom=10,
)

diagnostic_map.addLayer(
    vv_change_smooth,
    {
        "min": -8,
        "max": 8,
        "palette": [
            "b2182b",
            "f7f7f7",
            "2166ac",
        ],
    },
    "VV change (dB)",
)

diagnostic_map.addLayer(
    flood_candidate_mask,
    {
        "min": 0,
        "max": 1,
        "palette": ["00FFFF"],
    },
    "Flood-like candidate (−3 dB)",
)

diagnostic_map.addLayer(
    analysis_aoi,
    {
        "color": "yellow",
        "fillColor": "00000000",
    },
    "Analysis AOI",
)

for name, point in REFERENCE_POINTS.items():
    diagnostic_map.addLayer(
        ee.Feature(
            ee.Geometry.Point(
                [point["lon"], point["lat"]]
            ),
            {"name": name},
        ),
        {"color": "white"},
        name,
    )

diagnostic_map

Map(center=[28.200002497763045, 85.45000000000016], controls=(WidgetControl(options=['position', 'transparent_…

In [33]:
candidate_only_map = geemap.Map(
    center=AOI_CENTER,
    zoom=12,
)

candidate_only_map.addLayer(
    flood_candidate_mask,
    {
        "min": 0,
        "max": 1,
        "palette": ["00FFFF"],
        "opacity": 1,
    },
    "Flood-like candidate (−3 dB)",
)

candidate_only_map.addLayer(
    analysis_aoi,
    {
        "color": "yellow",
        "fillColor": "00000000",
    },
    "Analysis AOI",
)

candidate_only_map

Map(center=[28.200002497763045, 85.45000000000016], controls=(WidgetControl(options=['position', 'transparent_…

In [31]:
candidate_geometry = flood_candidate_mask.selfMask().geometry(
    maxError=10,
)

diagnostic_map = geemap.Map()

diagnostic_map.centerObject(candidate_geometry, 13)

diagnostic_map.addLayer(
    vv_change_smooth,
    {
        "min": -8,
        "max": 8,
        "palette": ["b2182b", "f7f7f7", "2166ac"],
    },
    "VV change (dB)",
)

diagnostic_map.addLayer(
    flood_candidate_mask,
    {
        "min": 0,
        "max": 1,
        "palette": ["00FFFF"],
    },
    "Flood-like candidate (−3 dB)",
)

diagnostic_map.addLayer(
    analysis_aoi,
    {
        "color": "yellow",
        "fillColor": "00000000",
    },
    "Analysis AOI",
)

diagnostic_map

Map(center=[28.20016612062262, 85.44971366265332], controls=(WidgetControl(options=['position', 'transparent_b…

In [34]:
candidate_only_map = geemap.Map(
    center=AOI_CENTER,
    zoom=13,
)

candidate_only_map.addLayer(
    flood_candidate_mask,
    {
        "min": 0,
        "max": 1,
        "palette": ["00FFFF"],
        "opacity": 1.0,
    },
    "Flood-like candidate (−3 dB)",
)

candidate_only_map.addLayer(
    analysis_aoi,
    {
        "color": "yellow",
        "fillColor": "00000000",
    },
    "Analysis AOI",
)

for name, point in REFERENCE_POINTS.items():
    candidate_only_map.addLayer(
        ee.Feature(
            ee.Geometry.Point(
                [point["lon"], point["lat"]]
            )
        ),
        {"color": "white"},
        name,
    )

candidate_only_map

Map(center=[28.200002497763045, 85.45000000000016], controls=(WidgetControl(options=['position', 'transparent_…